<a href="https://colab.research.google.com/github/purnimakushwaha/ITC101_Minor-project_python/blob/main/NASA_space_explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
#                    NASA SPACE EXPLORER
# ============================================================

import requests
import pandas as pd
import os
import time
import webbrowser
from datetime import datetime


# ============================================================
# API SETTINGS
# ============================================================

API_KEY = "A0xwnXUTvYvLGKK1zefTasGfcltlvHmz3eo0DXxH"

BASE_URL = "https://api.nasa.gov/planetary/apod"

FAVORITES_FILE = "nasa_favorites.csv"
HISTORY_FILE = "nasa_history.csv"


# ============================================================
# PROJECT DATA
# ============================================================

current_picture = None

favorites = []

history = []


# ============================================================
# API FUNCTION
# ============================================================

def get_nasa_picture(date=None):

    params = {
        "api_key": API_KEY
    }

    if date:
        params["date"] = date


    try:

        start_time = time.perf_counter()

        response = requests.get(
            BASE_URL,
            params=params,
            timeout=15
        )

        end_time = time.perf_counter()

        response_time = end_time - start_time


        print(
            f"\nAPI Response Time: "
            f"{response_time:.4f} seconds"
        )


        if response.status_code != 200:

            print(
                "\nNASA API request failed."
            )

            print(
                "HTTP Status:",
                response.status_code
            )

            return None


        data = response.json()

        return data


    except requests.exceptions.Timeout:

        print(
            "\nRequest timed out."
        )

        return None


    except requests.exceptions.ConnectionError:

        print(
            "\nInternet connection error."
        )

        return None


    except requests.exceptions.RequestException as error:

        print(
            "\nRequest Error:",
            error
        )

        return None


# ============================================================
# DISPLAY PICTURE
# ============================================================

def display_picture(data):

    if data is None:

        return


    print("\n" + "=" * 70)

    print(
        "                 NASA ASTRONOMY PICTURE"
    )

    print("=" * 70)


    print(
        "\nTitle:"
    )

    print(
        data.get(
            "title",
            "Not Available"
        )
    )


    print(
        "\nDate:"
    )

    print(
        data.get(
            "date",
            "Not Available"
        )
    )


    print(
        "\nMedia Type:"
    )

    print(
        data.get(
            "media_type",
            "Not Available"
        )
    )


    print(
        "\nExplanation:"
    )

    print(
        data.get(
            "explanation",
            "Not Available"
        )
    )


    print(
        "\nImage / Video URL:"
    )

    print(
        data.get(
            "url",
            "Not Available"
        )
    )


    if data.get("hdurl"):

        print(
            "\nHD Image URL:"
        )

        print(
            data["hdurl"]
        )


    print("=" * 70)


# ============================================================
# SAVE HISTORY
# ============================================================

def save_history(data):

    global history


    if data is None:

        return


    record = {

        "Date":
            data.get(
                "date",
                ""
            ),

        "Title":
            data.get(
                "title",
                ""
            ),

        "Media Type":
            data.get(
                "media_type",
                ""
            ),

        "URL":
            data.get(
                "url",
                ""
            ),

        "HD URL":
            data.get(
                "hdurl",
                ""
            ),

        "Checked At":
            datetime.now().strftime(
                "%d-%m-%Y %I:%M:%S %p"
            )
    }


    history.append(
        record
    )


# ============================================================
# TODAY'S PICTURE
# ============================================================

def todays_picture():

    global current_picture


    print("\n" + "=" * 70)

    print(
        "                  TODAY'S PICTURE"
    )

    print("=" * 70)


    current_picture = get_nasa_picture()


    if current_picture:

        display_picture(
            current_picture
        )

        save_history(
            current_picture
        )


# ============================================================
# SEARCH BY DATE
# ============================================================

def search_by_date():

    global current_picture


    print("\n" + "=" * 70)

    print(
        "                SEARCH BY DATE"
    )

    print("=" * 70)


    print(
        "\nEnter date in YYYY-MM-DD format."
    )

    print(
        "Example: 2025-08-10"
    )


    date = input(
        "\nEnter date: "
    ).strip()


    try:

        datetime.strptime(
            date,
            "%Y-%m-%d"
        )


    except ValueError:

        print(
            "\nInvalid date format."
        )

        print(
            "Use YYYY-MM-DD."
        )

        return


    print(
        "\nSearching NASA database..."
    )


    current_picture = get_nasa_picture(
        date
    )


    if current_picture:

        display_picture(
            current_picture
        )

        save_history(
            current_picture
        )


# ============================================================
# OPEN IMAGE / VIDEO
# ============================================================

def open_media():

    print("\n" + "=" * 70)

    print(
        "                  OPEN MEDIA"
    )

    print("=" * 70)


    if current_picture is None:

        print(
            "\nNo picture loaded."
        )

        print(
            "First choose Today's Picture "
            "or Search by Date."
        )

        return


    url = current_picture.get(
        "hdurl"
    )


    if not url:

        url = current_picture.get(
            "url"
        )


    if url:

        print(
            "\nOpening NASA media..."
        )

        webbrowser.open(
            url
        )

    else:

        print(
            "\nMedia URL not available."
        )


# ============================================================
# SAVE FAVORITE
# ============================================================

def save_favorite():

    global favorites


    print("\n" + "=" * 70)

    print(
        "                  SAVE FAVORITE"
    )

    print("=" * 70)


    if current_picture is None:

        print(
            "\nNo picture loaded."
        )

        return


    record = {

        "Date":
            current_picture.get(
                "date",
                ""
            ),

        "Title":
            current_picture.get(
                "title",
                ""
            ),

        "Media Type":
            current_picture.get(
                "media_type",
                ""
            ),

        "URL":
            current_picture.get(
                "url",
                ""
            ),

        "HD URL":
            current_picture.get(
                "hdurl",
                ""
            )
    }


    existing_dates = [

        item["Date"]

        for item in favorites
    ]


    if record["Date"] in existing_dates:

        print(
            "\n⭐ This picture is already saved."
        )

        return


    favorites.append(
        record
    )


    print(
        "\n⭐ Picture added to favorites!"
    )


# ============================================================
# VIEW FAVORITES
# ============================================================

def view_favorites():

    print("\n" + "=" * 70)

    print(
        "                   FAVORITES"
    )

    print("=" * 70)


    if len(favorites) == 0:

        print(
            "\nNo favorite pictures saved."
        )

        return


    for index, item in enumerate(
        favorites,
        start=1
    ):

        print(
            f"\n{index}. {item['Title']}"
        )

        print(
            "   Date:",
            item["Date"]
        )

        print(
            "   URL:",
            item["URL"]
        )


# ============================================================
# SAVE FAVORITES TO CSV
# ============================================================

def save_favorites():

    if len(favorites) == 0:

        print(
            "\nNo favorites to save."
        )

        return


    df = pd.DataFrame(
        favorites
    )


    df.to_csv(
        FAVORITES_FILE,
        index=False
    )


    print(
        "\nFavorites saved successfully!"
    )

    print(
        "File:",
        os.path.abspath(
            FAVORITES_FILE
        )
    )


# ============================================================
# LOAD FAVORITES
# ============================================================

def load_favorites():

    global favorites


    if not os.path.exists(
        FAVORITES_FILE
    ):

        return


    try:

        df = pd.read_csv(
            FAVORITES_FILE
        )


        favorites = (
            df.to_dict(
                orient="records"
            )
        )


        print(
            f"{len(favorites)} "
            "favorite(s) loaded."
        )


    except Exception as error:

        print(
            "Could not load favorites:",
            error
        )


# ============================================================
# VIEW HISTORY
# ============================================================

def view_history():

    print("\n" + "=" * 70)

    print(
        "                   SEARCH HISTORY"
    )

    print("=" * 70)


    if len(history) == 0:

        print(
            "\nNo history available."
        )

        return


    for index, item in enumerate(
        history,
        start=1
    ):

        print(
            f"\n{index}. "
            f"{item['Title']}"
        )

        print(
            "   Date:",
            item["Date"]
        )

        print(
            "   Checked:",
            item["Checked At"]
        )


# ============================================================
# EXPORT HISTORY
# ============================================================

def export_history():

    if len(history) == 0:

        print(
            "\nNo history available."
        )

        return


    try:

        df = pd.DataFrame(
            history
        )


        df.to_csv(
            HISTORY_FILE,
            index=False
        )


        print(
            "\nHistory exported successfully!"
        )

        print(
            "File:",
            os.path.abspath(
                HISTORY_FILE
            )
        )


    except Exception as error:

        print(
            "\nExport error:",
            error
        )


# ============================================================
# NASA PROJECT STATISTICS
# ============================================================

def show_statistics():

    print("\n" + "=" * 70)

    print(
        "                 PROJECT STATISTICS"
    )

    print("=" * 70)


    total_history = len(
        history
    )


    total_favorites = len(
        favorites
    )


    print(
        "\nPictures Viewed:",
        total_history
    )


    print(
        "Favorite Pictures:",
        total_favorites
    )


    if total_history > 0:

        media_types = {}

        for item in history:

            media_type = item[
                "Media Type"
            ]

            media_types[
                media_type
            ] = media_types.get(
                media_type,
                0
            ) + 1


        print(
            "\nMedia Types:"
        )


        for media_type, count in media_types.items():

            print(
                f"{media_type}: {count}"
            )


# ============================================================
# PROJECT INFORMATION
# ============================================================

def project_information():

    print("\n" + "=" * 70)

    print(
        "                 PROJECT INFORMATION"
    )

    print("=" * 70)


    print(
        "\nProject Name:"
    )

    print(
        "NASA Space Explorer"
    )


    print(
        "\nAPI Used:"
    )

    print(
        "NASA Astronomy Picture of the Day API"
    )


    print(
        "\nMain Python Concepts:"
    )

    print(
        "• REST API"
    )

    print(
        "• JSON"
    )

    print(
        "• requests"
    )

    print(
        "• Pandas"
    )

    print(
        "• CSV file handling"
    )

    print(
        "• Functions"
    )

    print(
        "• Exception handling"
    )

    print(
        "• Date handling"
    )

    print(
        "• Execution time"
    )


    print(
        "\nMain Features:"
    )

    print(
        "• Today's astronomy picture"
    )

    print(
        "• Search by date"
    )

    print(
        "• Open HD media"
    )

    print(
        "• Save favorites"
    )

    print(
        "• Search history"
    )

    print(
        "• CSV export"
    )

    print(
        "• Statistics"
    )


# ============================================================
# MAIN MENU
# ============================================================

def main_menu():

    while True:

        print("\n")

        print("=" * 70)

        print(
            "              🌌 NASA SPACE EXPLORER"
        )

        print("=" * 70)


        print(
            "1.  Today's Astronomy Picture"
        )

        print(
            "2.  Search Picture by Date"
        )

        print(
            "3.  Open Image / Video"
        )

        print(
            "4.  Save Current Picture as Favorite"
        )

        print(
            "5.  View Favorites"
        )

        print(
            "6.  Save Favorites to CSV"
        )

        print(
            "7.  View Search History"
        )

        print(
            "8.  Export History to CSV"
        )

        print(
            "9.  Project Statistics"
        )

        print(
            "10. Project Information"
        )

        print(
            "11. Exit"
        )


        print("=" * 70)


        choice = input(
            "Enter your choice: "
        ).strip()


        if choice == "1":

            todays_picture()


        elif choice == "2":

            search_by_date()


        elif choice == "3":

            open_media()


        elif choice == "4":

            save_favorite()


        elif choice == "5":

            view_favorites()


        elif choice == "6":

            save_favorites()


        elif choice == "7":

            view_history()


        elif choice == "8":

            export_history()


        elif choice == "9":

            show_statistics()


        elif choice == "10":

            project_information()


        elif choice == "11":

            print(
                "\n" + "=" * 70
            )

            print(
                "Thank you for using "
                "NASA Space Explorer! 🌌"
            )



            print("=" * 70)

            break


        else:

            print(
                "\nInvalid choice!"
            )

            print(
                "Please choose 1-11."
            )


# ============================================================
# START PROJECT
# ============================================================

print("=" * 70)

print(
    "                 🌌 NASA SPACE EXPLORER"
)

print(
    "                    PYTHON MINOR PROJECT"
)

print("=" * 70)


now = datetime.now()


print(
    "Date:",
    now.strftime(
        "%d-%m-%Y"
    )
)


print(
    "Time:",
    now.strftime(
        "%I:%M:%S %p"
    )
)


print("=" * 70)


load_favorites()


print(
    "\nProject started successfully!"
)

print(
    "Choose an option from the menu."
)


main_menu()

                 🌌 NASA SPACE EXPLORER
                    PYTHON MINOR PROJECT
Date: 10-08-2026
Time: 06:34:05 PM

Project started successfully!
Choose an option from the menu.


              🌌 NASA SPACE EXPLORER
1.  Today's Astronomy Picture
2.  Search Picture by Date
3.  Open Image / Video
4.  Save Current Picture as Favorite
5.  View Favorites
6.  Save Favorites to CSV
7.  View Search History
8.  Export History to CSV
9.  Project Statistics
10. Project Information
11. Exit
Enter your choice: 1

                  TODAY'S PICTURE

API Response Time: 9.7645 seconds

NASA API request failed.
HTTP Status: 503


              🌌 NASA SPACE EXPLORER
1.  Today's Astronomy Picture
2.  Search Picture by Date
3.  Open Image / Video
4.  Save Current Picture as Favorite
5.  View Favorites
6.  Save Favorites to CSV
7.  View Search History
8.  Export History to CSV
9.  Project Statistics
10. Project Information
11. Exit
Enter your choice: 1

                  TODAY'S PICTURE

API Response Time: 